## About Dataset
Context This is a small subset of dataset of Book reviews from Amazon Kindle Store category.

Content 5-core dataset of product reviews from Amazon Kindle Store category from May 1996 - July 2014. Contains total of 982619 entries. Each reviewer has at least 5 reviews and each product has at least 5 reviews in this dataset. Columns

- **asin**: ID do produto, como B000FA64PK  
- **helpful**: avaliação de utilidade da review – exemplo: 2/3  
- **overall**: avaliação do produto  
- **reviewText**: texto da review (título)  
- **reviewTime**: data da review (bruta)  
- **reviewerID**: ID do revisor, como A3SPTOKDG7WBLN  
- **reviewerName**: nome do revisor  
- **summary**: resumo da review (descrição)  
- **unixReviewTime**: timestamp unix  
- **Acknowledgements**: Este dataset foi retirado dos dados de produtos da Amazon, Julian McAuley, site da UCSD. http://jmcauley.ucsd.edu/data/amazon/

License to the data files belong to them.

### Inspiration

- Sentiment analysis on reviews.
- Understanding how people rate usefulness of a review / What factors influence helpfulness of a review.
- Fake reviews / outliers.
- Best rated product IDs, or similarity between products based on reviews alone (not the best idea ikr).
- Any other interesting analysis

### Best Practises
- Preprocessing And Cleaning
- Train Test Split
- BOW,TFIDF,Word2vec
- Train ML algorithms

In [17]:
import pandas as pd
import numpy as np

import re
import nltk
from nltk.corpus import stopwords
from bs4 import BeautifulSoup

from nltk.stem import WordNetLemmatizer

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report

nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\welli\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [18]:
## load the dataset
df = pd.read_csv("https://raw.githubusercontent.com/krishnaik06/Complete-Data-Science-With-Machine-Learning-And-NLP-2024/refs/heads/main/26-CompleteNLP%20For%20Machine%20Learning/Practicals/Kindle%20Reviews/all_kindle_review.csv")
df.head()

,Unnamed: 0.1,Unnamed: 0,asin,helpful,rating,reviewText,reviewTime,reviewerID,reviewerName,summary,unixReviewTime
0,0,11539,B0033UV8HI,"[8, 10]",3,"Jace Rankin may be short, but he's nothing to ...","09 2, 2010",A3HHXRELK8BHQG,Ridley,Entertaining But Average,1283385600
1,1,5957,B002HJV4DE,"[1, 1]",5,Great short read. I didn't want to put it dow...,"10 8, 2013",A2RGNZ0TRF578I,Holly Butler,Terrific menage scenes!,1381190400
2,2,9146,B002ZG96I4,"[0, 0]",3,I'll start by saying this is the first of four...,"04 11, 2014",A3S0H2HV6U1I7F,Merissa,Snapdragon Alley,1397174400
3,3,7038,B002QHWOEU,"[1, 3]",3,Aggie is Angela Lansbury who carries pocketboo...,"07 5, 2014",AC4OQW3GZ919J,Cleargrace,very light murder cozy,1404518400
4,4,1776,B001A06VJ8,"[0, 1]",4,I did not expect this type of book to be in li...,"12 31, 2012",A3C9V987IQHOQD,Rjostler,Book,1356912000


In [19]:
df = df[['reviewText', 'rating']]
df.head()

,reviewText,rating
0,"Jace Rankin may be short, but he's nothing to ...",3
1,Great short read. I didn't want to put it dow...,5
2,I'll start by saying this is the first of four...,3
3,Aggie is Angela Lansbury who carries pocketboo...,3
4,I did not expect this type of book to be in li...,4


In [20]:
df.shape

(12000, 2)

In [21]:
## Missing Values
df.isnull().sum()

reviewText    0
rating        0
dtype: int64

In [22]:
df['rating'].unique()

array([3, 5, 4, 2, 1], dtype=int64)

In [23]:
df['rating'].value_counts()

rating
5    3000
4    3000
3    2000
2    2000
1    2000
Name: count, dtype: int64

### Preprocessing and Cleaning

In [24]:
## positive review is 1 and negative review is 0

df['rating'] = df['rating'].apply(lambda x: 0 if x < 3 else 1)

In [25]:
df['rating'].value_counts()

rating
1    8000
0    4000
Name: count, dtype: int64

In [26]:
## 1. Lower All the cases
df['reviewText'] = df['reviewText'].str.lower()

In [27]:
df.head()

,reviewText,rating
0,"jace rankin may be short, but he's nothing to ...",1
1,great short read. i didn't want to put it dow...,1
2,i'll start by saying this is the first of four...,1
3,aggie is angela lansbury who carries pocketboo...,1
4,i did not expect this type of book to be in li...,1


In [31]:
## Removing special characters
df['reviewText'] = df['reviewText'].apply(lambda x: re.sub('[^a-z A-z 0-9-]+', '', x))

## Remove the stopwords
df['reviewText'] = df['reviewText'].apply(lambda x: " ".join([y for y in x.split() if y not in stopwords.words('english')]))

## Remove url
df['reviewText'] = df['reviewText'].apply(lambda x:  re.sub(r'(http|https|ftp|ssh)://([\w_-]+(?:(?:\.[\w_-]+)+))([\w.,@?^=%&:/~+#-]*[\w@?^=%&/~+#-])?', '' , str(x)))

## Remove html tags
# df['reviewText']=df['reviewText'].apply(lambda x: BeautifulSoup(x, 'lxml').get_text())

## Remove any additional spaces
df['reviewText']=df['reviewText'].apply(lambda x: " ".join(x.split()))

In [33]:
## Lemmatizer
lemmatizer = WordNetLemmatizer()

def lemmatize_words(text):
    return " ".join([lemmatizer.lemmatize(word) for word in text.split()])

df['reviewText'] = df['reviewText'].apply(lambda x: lemmatize_words(x))

In [34]:
df.head()

,reviewText,rating
0,jace rankin may short he nothing mess man haul...,1
1,great short read didnt want put read one sitti...,1
2,ill start saying first four book wasnt expecti...,1
3,aggie angela lansbury carry pocketbook instead...,1
4,expect type book library pleased find price right,1


### Train Test Split

In [36]:
X_train, X_test, y_train, y_test = train_test_split(df['reviewText'], df['rating'], test_size=0.20)

### BOW and TF-IDF

In [37]:
bow = CountVectorizer()

X_train_bow = bow.fit_transform(X_train).toarray()
X_test_bow = bow.transform(X_test).toarray()

In [38]:
tfidf = TfidfVectorizer()

X_train_tfidf = tfidf.fit_transform(X_train).toarray()
X_test_tfidf = tfidf.transform(X_test).toarray()

In [39]:
X_train_bow

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]], dtype=int64)

In [40]:
nb_model_bow = GaussianNB().fit(X_train_bow, y_train)
nb_model_tfidf = GaussianNB().fit(X_train_tfidf, y_train)

In [41]:
y_pred_bow = nb_model_bow.predict(X_test_bow)
y_pred_tfidf = nb_model_bow.predict(X_test_tfidf)

confusion_matrix(y_test, y_pred_bow)

array([[496, 300],
       [711, 893]], dtype=int64)

In [42]:
print('BOW accuracy: ', accuracy_score(y_test, y_pred_bow))

BOW accuracy:  0.57875


In [43]:
confusion_matrix(y_test, y_pred_tfidf)

array([[487, 309],
       [707, 897]], dtype=int64)

In [44]:
print('TFIDF accuracy: ', accuracy_score(y_test, y_pred_tfidf))

TFIDF accuracy:  0.5766666666666667
